# Synthesis — MR to synthetic CT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fideus-labs/KonfAI/blob/main/examples/Synthesis/Synthesis_demo.ipynb)

**Run all the cells.** This notebook trains a UNet++ generator to turn an MR volume into a synthetic
CT, predicts it, and scores it with MAE / PSNR / SSIM inside the body mask.

Nothing is gated behind a flag. Expect roughly **7 minutes on a GPU** (on Colab pick
*Runtime > Change runtime type > GPU*).

In [ ]:
# Setup: find KonfAI (cloning it on Colab), install what is missing, load the notebook helpers.
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/fideus-labs/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples").is_dir())
sys.path.insert(0, str(REPO_DIR / "examples"))

from konfai_demo import latest_checkpoint, read, run, setup, show

EXAMPLE_DIR, DATASET_DIR, DEVICE = setup(REPO_DIR, "Synthesis", ("konfai", f"{REPO_DIR}[imaging,smp,ssim]"), "huggingface_hub", "matplotlib", ("segmentation_models_pytorch", "segmentation-models-pytorch"))

## 1. The data

Two paired head-and-neck / thorax cases (~59 MB), three volumes each:

```text
Dataset/1HNA001/MR.mha    # model input
Dataset/1HNA001/CT.mha    # target
Dataset/1HNA001/MASK.mha  # body mask, used for preprocessing and for masked scoring
```

In [ ]:
import shutil

from huggingface_hub import snapshot_download

if not any(DATASET_DIR.glob("*/MR.mha")):
    snapshot_download("VBoussot/konfai-demo", repo_type="dataset", allow_patterns="Synthesis/**", local_dir=str(DATASET_DIR))
    for case in (DATASET_DIR / "Synthesis").iterdir():
        shutil.move(str(case), DATASET_DIR / case.name)
    shutil.rmtree(DATASET_DIR / "Synthesis")
    shutil.rmtree(DATASET_DIR / ".cache", ignore_errors=True)

CASES = sorted(path.name for path in DATASET_DIR.iterdir() if path.is_dir())
print(len(CASES), "cases:", ", ".join(CASES))

## 2. Look at one case

In [ ]:
import numpy as np

case = DATASET_DIR / CASES[0]
mr, ct, mask = (read(case / f"{name}.mha") for name in ("MR", "CT", "MASK"))
print(case.name, "| shape:", mr.shape, "| CT range:", (int(ct.min()), int(ct.max())), "HU")

BEST = int(np.argmax(mask.sum(axis=(1, 2))))  # the slice with the most body voxels
show([(f"MR — slice {BEST}", mr[BEST], "gray"), ("CT (target)", ct[BEST], "gray"), ("MASK", mask[BEST], "viridis")])

## 3. Train, predict, evaluate

| Command | Config | What it does |
|---|---|---|
| `konfai TRAIN` | `Config.yml` | trains `UNetpp5` (from `Model.py`) with an MAE + perceptual loss |
| `konfai PREDICTION` | `Prediction.yml` | 2.5D slice-wise inference with flip TTA, writes `Predictions/TRAIN_01/` |
| `konfai EVALUATION` | `Evaluation.yml` | MAE / PSNR / SSIM inside `MASK` |

The training is deliberately short so the notebook finishes; raise `epochs` in `Config.yml` for a real
run. `Config_GAN.yml` trains the same generator adversarially against a 3D discriminator — see the
README.

In [ ]:
run("konfai", "TRAIN", "-y", *DEVICE, "--config", "Config.yml")

run("konfai", "PREDICTION", "-y", *DEVICE, "--config", "Prediction.yml", "--models", latest_checkpoint("TRAIN_01"))
run("konfai", "EVALUATION", "-y", "--config", "Evaluation.yml")

## 4. The result

The synthetic CT next to the real one, on the same HU window, and the error between them. One epoch
already gets to about **98 HU** of mean absolute error inside the mask, with the residual concentrated
on bone edges and on the sharpest air/tissue boundaries — the places where a slice-wise model has the
least context.

The scores come out this way only because `Prediction.yml` preprocesses the MR exactly as `Config.yml`
did. Standardizing inside the mask at prediction time instead — a reasonable-looking change — feeds the
network a scale it never trained on and costs 4x on MAE.

In [ ]:
import json

import numpy as np

metrics = json.loads((EXAMPLE_DIR / "Evaluations" / "TRAIN_01" / "Metric_TRAIN.json").read_text())
for name, values in metrics["aggregates"].items():
    print(f"{name:24s} mean {values['mean']:.3f}   min {values['min']:.3f}   max {values['max']:.3f}")

synthetic = read(EXAMPLE_DIR / "Predictions" / "TRAIN_01" / "Dataset" / case.name / "sCT.mha")
difference = np.abs(synthetic.astype(np.float32) - ct.astype(np.float32)) * (mask > 0)
window = (-500.0, 1000.0)  # the same HU window on both, otherwise they cannot be compared by eye
show([
    ("real CT", ct[BEST], "gray", window),
    ("synthetic CT", synthetic[BEST], "gray", window),
    ("|difference| (HU)", difference[BEST], "magma", (0.0, 400.0)),
])

## What to change next

- **more training** — raise `epochs` in `Config.yml`.
- **the GAN variant** — `konfai TRAIN -y --config Config_GAN.yml` trains the same `UNetpp5` generator
  against a 3D discriminator; point `train_name` at `TRAIN_GAN_01` in `Prediction.yml` to reuse it.
- **your own pairs** — adapt `dataset_filenames` and the `MR` / `CT` group names.

`README.md` in this folder walks through the config, the `;accu;` patch semantics, and the two
patching scopes of the GAN.